# Lekce 11 - Protokol agent-agent (A2A)


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Co je protokol A2A?

Tento **Agent-to-Agent (A2A) protokol** je otevřený standard, který umožňuje AI agentům komunikovat, navzájem se objevovat a spolupracovat — i když jsou postaveni na různých rámcích nebo hostováni různými službami.

Key concepts:

- **Discovery** – Agenti zveřejňují *kartu agenta*, která popisuje jejich schopnosti, což usnadňuje ostatním agentům (nebo orchestrátorům) najít pro úkol správného specialistu.
- **Message Passing** – Agenti si vyměňují strukturované zprávy prostřednictvím společného protokolu, takže požadavek od jednoho agenta může být pochopen a splněn jiným bez ohledu na jeho vnitřní implementaci.
- **Task Lifecycle** – A2A definuje stavy jako *submitted*, *working*, *completed*, and *failed*, což dává orchestrátorovi plný přehled o tom, jak postupuje delegovaný úkol.

V této lekci simulujeme spolupráci ve stylu A2A tím, že propojíme tři specializované cestovní agenty do pracovního postupu, kde každý agent přispívá svými odbornými znalostmi a předává výsledky dalšímu.


## Vytváření specializovaných cestovních agentů


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Spolupráce více agentů prostřednictvím pracovního postupu

Propojujeme tři agenty do sekvenčního pracovního postupu, který odráží A2A přenos zpráv:

1. **CurrencyExchangeAgent** obdrží uživatelský požadavek a poskytne pokyny týkající se měny.
2. **ActivityPlannerAgent** obdrží obohacený kontext a přidá doporučení aktivit.
3. **TravelManagerAgent** syntetizuje oba vstupy do konečného cestovního přehledu.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Porozumění A2A v produkčním prostředí

V produkčním prostředí protokol A2A odemyká silné scénáře napříč službami:

| Schopnost | Popis |
|---|---|
| **Interoperabilita mezi frameworky** | Agent postavený v jednom frameworku může delegovat úkoly agentu vytvořenému v libovolném jiném frameworku kompatibilním s A2A, což umožňuje skutečnou interoperabilitu napříč organizacemi. |
| **Hranice služeb** | Agenti mohou běžet v samostatných mikroslužbách, cloudových regionech nebo dokonce v různých organizacích a přesto spolu bezproblémově spolupracovat. |
| **Dynamické objevování** | Orchestrátor může za běhu dotazovat registr Agent Card a najít nejvhodnějšího specialistu pro daný dílčí úkol. |
| **Streaming a push notifikace** | A2A podporuje Server-Sent Events (SSE) pro aktualizace průběhu v reálném čase a push notifikace pro dlouhotrvající úlohy. |

Pracovní postup, který jsme výše vytvořili, je zjednodušenou verzí tohoto vzoru běžící v rámci procesu. V reálném
nasazení by každý agent vystavoval HTTP endpoint, publikoval Agent Card a komunikoval
pomocí protokolu A2A JSON-RPC.


## Shrnutí

V této lekci jste se naučili:

1. **Co je protokol A2A** — otevřený standard pro objevování agentů mezi sebou, zasílání zpráv,
   a správu úkolů.
2. **Jak vytvořit specializované agenty** — agent pro směnu měn, agent plánovače aktivit,
   a orchestrátor pro správu cestování.
3. **Jak zapojit agenty do pracovního postupu** — pomocí `WorkflowBuilder` modelovat sekvenční
   předávání zpráv mezi agenty.
4. **Jak A2A funguje v produkci** — umožnění spolupráce napříč frameworky a službami
   s dynamickým objevováním a streamovanými aktualizacemi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Prohlášení o vyloučení odpovědnosti:
Tento dokument byl přeložen pomocí AI překladatelské služby Co‑op Translator (https://github.com/Azure/co-op-translator). I když usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho původním jazyce by měl být považován za závazný zdroj. Pro zásadní informace se doporučuje profesionální překlad provedený lidským překladatelem. Nejsme odpovědní za jakákoli nedorozumění nebo mylné výklady vzniklá v důsledku použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
